# Hands-On 4: Build a Tool-Calling Agent

The agent binds tools to a model and routes tool calls automatically: Tools → Model → ToolNode → Tool Result → Model.

#### Imports

We import `tool` for creating tools, `ToolNode` for executing tool calls, `MessagesState` for message state, and `ChatOpenAI` for the model.

In [1]:
%pip install -U langchain-openai

   ---------------------------------------- 0.0/570.0 kB ? eta -:--:--
   ---------------------------------------- 570.0/570.0 kB 5.8 MB/s  0:00:00

  Attempting uninstall: langchain-core

    Found existing installation: langchain-core 1.5.6

    Uninstalling langchain-core-1.5.6:

      Successfully uninstalled langchain-core-1.5.6

   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
  Attempting 


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install -U python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import os
from dotenv import load_dotenv

from typing import Annotated, Literal

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

In [30]:
load_dotenv(r"D:\Agent_Flow\.env", override=True)

print("OpenRouter key loaded:", bool(os.getenv("OPENROUTER_API_KEY")))

OpenRouter key loaded: True


#### Create Calculator Tool

A tool is a Python function with a docstring. The docstring becomes the schema that the model uses to call the tool.

In [32]:
@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression."""
    try:
        allowed = "0123456789+-*/(). "
        
        if not all(char in allowed for char in expression):
            return "Invalid expression."
        
        result = eval(expression)
        return str(result)
    
    except Exception as e:
        return f"Calculation error: {e}"

In [33]:
print(calculator.invoke("42 * 8"))

336


#### Create Search-Style Tool

A simple fake search tool that answers from a small knowledge base.

In [34]:
@tool
def search_docs(query: str) -> str:
    """Search internal documents for information."""
    
    knowledge = {
        "rag": "RAG stands for Retrieval-Augmented Generation. It retrieves relevant information from external documents and provides it to an LLM to generate a grounded response.",
        "langgraph": "LangGraph is a framework for building stateful, multi-step agent workflows.",
        "tool calling": "Tool calling allows an LLM to select and invoke external tools based on the user's request."
    }
    
    query_lower = query.lower()
    
    for key, value in knowledge.items():
        if key in query_lower:
            return value
    
    return f"No information found for: {query}"

In [35]:
print(search_docs.invoke("What is RAG?"))

RAG stands for Retrieval-Augmented Generation. It retrieves relevant information from external documents and provides it to an LLM to generate a grounded response.


### Create Tools List

In [36]:
tools = [calculator, search_docs]

print([tool.name for tool in tools])

['calculator', 'search_docs']


#### Bind Tools to Model

We attach the tools to the LLM using `bind_tools`. This allows the model to request tool calls.

#### Create OpenRouter LLM

In [37]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

print("LLM created successfully")

LLM created successfully


In [38]:
llm_with_tools = llm.bind_tools(tools)

print("Tools bound to model")

Tools bound to model


In [39]:
response = llm_with_tools.invoke(
    "Calculate 42 * 8"
)

print(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 67, 'total_tokens': 84, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 2.025e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 2.025e-05, 'upstream_inference_prompt_cost': 1.005e-05, 'upstream_inference_completions_cost': 1.02e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_6339e52f56', 'id': 'gen-1787215559-IlxehrzU42zyUUMbHPEd', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a01e59-3f84-7161-b3d4-8f66a324b372-0' tool_calls=[{'name': 'calculator', 'args': {'expression': '42 * 8'}, 'id': 'c

In [40]:
print(response.tool_calls)

[{'name': 'calculator', 'args': {'expression': '42 * 8'}, 'id': 'call_W4IyhJgGyc16Srj0wfcWbVwU', 'type': 'tool_call'}]


### create toolNode

In [41]:
tool_node = ToolNode(tools)

print("ToolNode created successfully")

ToolNode created successfully


#### Add ToolNode

`ToolNode` executes the tool calls that the model requests.

In [42]:
def call_model(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

### create Graph

In [43]:
builder = StateGraph(MessagesState)

builder.add_node("agent", call_model)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")

#### Model Node + Router

The model node calls the LLM. The router then checks if the last message has tool calls that need to be executed.

In [44]:
builder.add_conditional_edges(
    "agent",
    tools_condition,
    {
        "tools": "tools",
        END: END
    }
)

builder.add_edge("tools", "agent")

### compile agent

In [45]:
agent = builder.compile()

print("Agent compiled successfully")

Agent compiled successfully


#### Test: Simple Math Query

Invoke the agent with "Calculate 42 * 8". The model should call the calculator tool, get the result, and return it.

In [46]:
result = agent.invoke({
    "messages": [
        HumanMessage(content="Calculate (45 / 5) + (12 * 3)")
    ]
})

for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}")

human: Calculate (45 / 5) + (12 * 3)
ai: 
tool: 45.0
ai: The result of the calculation \((45 / 5) + (12 * 3)\) is \(45.0\).


#### Test: Search Query

Invoke the agent with "What is RAG?" to verify it queries the search docs tool.

In [47]:
result = agent.invoke({
    "messages": [
        HumanMessage(content="What is RAG?")
    ]
})

for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}")

human: What is RAG?
ai: 
tool: RAG stands for Retrieval-Augmented Generation. It retrieves relevant information from external documents and provides it to an LLM to generate a grounded response.
ai: RAG stands for Retrieval-Augmented Generation. It is a technique that retrieves relevant information from external documents and provides it to a language model (LLM) to generate a grounded response.


#### Test: Multi-Step Tool Call

The model may chain multiple tool calls before reaching a final answer, as shown with a compound arithmetic query.

In [48]:
result = agent.invoke({
    "messages": [
        HumanMessage(
            content="What is RAG and calculate 45 * 12"
        )
    ]
})

for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}")

human: What is RAG and calculate 45 * 12
ai: 
tool: RAG stands for Retrieval-Augmented Generation. It retrieves relevant information from external documents and provides it to an LLM to generate a grounded response.
tool: 540
ai: RAG stands for Retrieval-Augmented Generation. It retrieves relevant information from external documents and provides it to a language model to generate a grounded response.

The calculation of \( 45 \times 12 \) is \( 540 \).


#### Summary

The model decides when to call tools, ToolNode executes them, and the results loop back to the model until a final answer is produced.